# RETFound → combined_dr — the generalization fix

**Why this notebook exists.** Our APTOS-only RETFound scored κ≈0.95 on APTOS but
collapsed to κ≈0.59 / acc≈0.33 on IDRiD (a dataset no model had seen), dumping ~46% of
images into *Mild* and missing most severe disease. That is a **single-source domain-shift**
failure, not under-training. The fix is to retrain on the **multi-source** set
(EyePACS+APTOS+Messidor) with imbalance handling, then re-check on IDRiD.

This trains **RETFound on `combined_dr`** with the fix recipe already in the repo:
focal loss, balanced sampler, label-smoothed CE fallback, `backbone_lr_scale=0.5`.

### Prerequisites (do these first)
1. **Accelerator = GPU T4 ×2**, and we pass `train.devices=1` (single T4). **Never P100** — Kaggle's torch lacks sm_60 kernels. See `docs/kaggle-notebook.md` §6.
2. **Internet = ON** (Settings → Internet) — needed for pretrained + RETFound weight downloads and the IDRiD external check.
3. **RETFound weights are GATED.** Visit https://huggingface.co/YukunZhou/RETFound_mae_natureCFP and accept access with your HF account.
4. Add your HF token as a Kaggle **Secret** named `HF_TOKEN` (Add-ons → Secrets).
5. **Add Input**: the merged `eyepacs-aptos-messidor-diabetic-retinopathy` dataset.


## 1. Install the package


In [ ]:
!pip install -q -e .
!pip install -q -r requirements.txt
# NOTE: don't let this downgrade Kaggle's CUDA torch. If it does, reinstall the Kaggle wheel.

## 2. Authenticate to Hugging Face (gated RETFound weights)
If this fails, you either haven't accepted the license (prereq #3) or the `HF_TOKEN` secret is missing/misnamed.


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret('HF_TOKEN'))
print('HF login OK')

## 3. Point at the merged dataset & sanity-check it's mounted


In [ ]:
import os
ROOT = '/kaggle/input/eyepacs-aptos-messidor-diabetic-retinopathy'
assert os.path.isdir(ROOT), f'Not mounted: {ROOT} — use + Add Input to attach it.'
print('top-level entries:', sorted(os.listdir(ROOT))[:20])

## 4. Build the combined_dr manifest
Auto-detects class-folder vs CSV labels, does a stratified train/val/test split, and prints the
class distribution so you can eyeball it before a long run. Written to the writable `/kaggle/working`.


In [ ]:
!python scripts/prepare_combined_dr.py \
    --root {ROOT} \
    --manifest /kaggle/working/combined_dr_manifest.csv

## 5. Resume control (for the 12h session cap)

`combined_dr` is ~92k images — RETFound (ViT-L) will very likely need **more than one 12h
session**. The plan:

- Each session stops cleanly at `max_time` (11h30), leaving `last.ckpt` intact.
- **`/kaggle/working` is wiped between sessions**, so to continue you must persist the
  checkpoints: after a session, *Save Version* (commit) or create a Kaggle **Dataset** from
  `/kaggle/working/outputs`. Next session, **Add Input** that dataset and set `RESUME_CKPT`
  below to its `last.ckpt` path.

First session: leave `RESUME_CKPT = None`.


In [ ]:
# First session: None. Later sessions: e.g.
# RESUME_CKPT = '/kaggle/input/<your-saved-output-dataset>/outputs/combined_dr/retfound/last.ckpt'
RESUME_CKPT = None

## 6. Train RETFound on combined_dr (single T4, 12h-safe)

Uses the ready `+experiment=retfound_combined` (model=retfound, data=combined_dr, image_size=224,
`backbone_lr_scale=0.5`, `loss=focal`, `max_epochs=30`). We add the Kaggle-safe overrides:
single GPU, `num_workers=4`, and `max_time` to stop before the 12h wall. Balanced sampling is on by default.

If you hit CUDA OOM on the T4, drop `train.batch_size=8`.


In [ ]:
import subprocess
cmd = [
    'python', '-m', 'occuwise.train', '+experiment=retfound_combined',
    f'data.data_root={ROOT}',
    'data.manifest=/kaggle/working/combined_dr_manifest.csv',
    'train.devices=1', 'train.num_workers=4', 'train.batch_size=16',
    'train.max_time=00:11:30:00',
]
if RESUME_CKPT:
    cmd.append(f'train.resume={RESUME_CKPT}')
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)

## 7. Locate the checkpoints (persist these before the session ends)
`best-*.ckpt` = best `val/quadratic_kappa`; `last.ckpt` = resume point. Save `/kaggle/working/outputs`
as a Version/Dataset so you can resume or download.


In [ ]:
!ls -la /kaggle/working/outputs/combined_dr/retfound/ 2>/dev/null || echo 'no checkpoints yet'

## 8. The honest test — external validation on IDRiD

IDRiD is **not** in `combined_dr`, so this is the unbiased generalization number — the same
metric we used to expose the original collapse. Compare against the APTOS-only baseline
(κ 0.594 / acc 0.33 / macro-F1 0.31). We want κ up **and** the *Mild*-dumping gone (recall
restored on grades 2–4).


In [ ]:
import io, glob, numpy as np, pandas as pd, torch
from PIL import Image
from huggingface_hub import hf_hub_download
import sys; sys.path.insert(0, 'src')
from occuwise.data.transforms import build_transforms
from occuwise.engine.lit_classifier import LitClassifier
from occuwise.engine.metrics import classification_metrics

ckpts = sorted(glob.glob('/kaggle/working/outputs/combined_dr/retfound/best-*.ckpt'))
assert ckpts, 'No best checkpoint found — train first.'
CKPT = ckpts[-1]; print('using', CKPT)

# Build arch offline (weights come from the checkpoint, not the gated download).
ck = torch.load(CKPT, map_location='cpu', weights_only=False)
hp = dict(ck['hyper_parameters']); hp.update(weights_repo=None, weights_file=None, pretrained=False)
model = LitClassifier(**hp); model.load_state_dict(ck['state_dict']); model.eval()
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; model.to(dev)

# IDRiD (516 images) straight from the HF parquet — no manual download.
parts = []
for split, h in [('train','d81b05cdbfbe95cd'), ('test','3eaaa0286c6f240e')]:
    p = hf_hub_download('amin-nejad/idrid-disease-grading', f'data/{split}-00000-of-00001-{h}.parquet', repo_type='dataset')
    parts.append(pd.read_parquet(p))
df = pd.concat(parts, ignore_index=True)

tf = build_transforms('classification', 'fundus', 224, train=False)
met = classification_metrics(5)
yt, yp = [], []
with torch.no_grad():
    for i in range(0, len(df), 32):
        xb = [tf(image=np.array(Image.open(io.BytesIO(b['bytes'])).convert('RGB')))['image'] for b in df['image'].iloc[i:i+32]]
        x = torch.stack(xb).to(dev)
        probs = torch.softmax(model(x), 1)
        y = torch.tensor(df['label'].iloc[i:i+len(xb)].to_numpy())
        met.update(probs.cpu(), y)
        yp += probs.argmax(1).cpu().tolist(); yt += y.tolist()

res = {k.split('/')[-1]: round(float(v),4) for k,v in met.compute().items()}
print('IDRiD external:', res)
yt, yp = np.array(yt), np.array(yp)
print('pred dist:', {i:int((yp==i).sum()) for i in range(5)}, ' (true:', {i:int((yt==i).sum()) for i in range(5)}, ')')
cm = np.zeros((5,5),int)
for t,p in zip(yt,yp): cm[t,p]+=1
print('confusion (rows=true, cols=pred):')
for i in range(5): print(f'  t={i}', cm[i], f'recall={cm[i,i]/max(cm[i].sum(),1):.2f}')

---
### If IDRiD κ is still weak after this
The next levers (in order): **stronger cross-camera augmentation** (widen brightness/contrast/gamma/hue
in `src/occuwise/data/transforms.py`), an **ordinal head** (CORAL/CORN) so errors stay adjacent, and
**post-hoc logit/prior adjustment** to correct residual class-prior skew. Multi-source data is the big
lever though — this run tests it first.
